# Using IfcOpenShell in Python

A short introduction to reading and inspecting IFC model data with Python.

## 1. Setup

To use the IfcOpenShell, first install the packages with `python -m pip install ifcopenshell pandas jupyter` and import them below.


In [ ]:
from pathlib import Path # Path finding python extension
from collections import Counter

import ifcopenshell # Imports the newest available IfcOpenShell version
import ifcopenshell.util.element
import pandas as pd

print('IfcOpenShell version:', ifcopenshell.version) # Returns and prints the users current version of IfcOpenShell

IfcOpenShell version: 0.8.5


**Infamous bug:** When installing a package, it is common to run the command `pip install <package>`. This is not good practise, as when the command is run without specifying the environment, Python searches your systems global `PATH` variable and picks the first `pip` executable it find. 

If you have multiple Python versions (e.g. Python 3.10, 3.12, virtual environment etc.), the global `pip` shortcut might point to a completely different Python version than the `python` command you are currently using. 

This causes an infamous bug, where a package says **"Succesfully installed"** in the terminal, but when you run your script, you get a `ModuleNotFoundError`. 

Therefore, you need to run the command `python -m pip install <package>`

## 2. Open an IFC file

To open an IFC file in python, the following code takes the path given by the user, and runs it to open the file via. IfcOpenShell.

Set the path to your own IFC file.

In [ ]:
IFC_PATH = Path('example.ifc')  # Here the input should be change to your own .ifc model path

if not IFC_PATH.is_file(): # Error code to announce issues with the input file in above function.
    raise FileNotFoundError(
        f'Could not find {IFC_PATH.resolve()}. Update IFC_PATH to an existing .ifc file.'
    )

model = ifcopenshell.open(str(IFC_PATH))
print('IFC schema:', model.schema) # Prints the imported file's IFC schema noted, to differ between newer or older IFC languages/systems.
print('Total entities:', len(list(model))) # Returns the amount of entities in the loaded IFC file.

IFC schema: IFC4X3
Total entities: 362197


## 3. Find and count elements

For further analaysis of the data in the loaded IFC file, the `by_type()` function can be utilized, as it retrieves elements of an IFC class.

Utilizing this function, a short code can be run to count the amount of elements with the defined classes by the user, are present in the IFC file.

In [ ]:
classes = ['IfcWall', 'IfcWindow', 'IfcDoor', 'IfcSpace', 'IfcBuildingStorey'] # Here, the user can specifiy which IFC classes to further analyze.

class_counts = pd.DataFrame({ # A function which takes the defined "classes" and counts their apparence in the file.
    'IFC class': classes,
    'Count': [len(model.by_type(cls)) for cls in classes]
})
display(class_counts) # Prints a table oversight of the above data collected.

,IFC class,Count
0,IfcWall,149
1,IfcWindow,229
2,IfcDoor,101
3,IfcSpace,3
4,IfcBuildingStorey,6


## 4. Read properties

IfcOpenSchell can be used to list the attributes belonging to an IfcClass element, such as its "Qto_" or "Pset_" sets and read their documented values, units and hyperboles etc.

The attribute type properties that can be read are for example direct attributes, that include `Name` and `GlobalId`. 
Additional values may be in corresponding property sets.

In [ ]:
ELEMENT_CLASS = 'IfcWall'  # Try IfcWindow, IfcDoor or another populated class that requires investigation.
elements = model.by_type(ELEMENT_CLASS) # Simplifying the elements defined by the user.

if not elements: # If no correct element type is named, error message is given.
    print(f'No {ELEMENT_CLASS} elements found. Choose another populated class.')
else:
    element = elements[0] # First instance in elements list
    print('Class:', element.is_a()) # returns the attributes class
    print('Name:', element.Name) # returns the name of the element
    print('GlobalId:', element.GlobalId) # returns the elements uniqe ID within tthe IFC project file
    psets = ifcopenshell.util.element.get_psets(element)
    print('Available property / quantity sets:', list(psets)) # lists the corresponding psets/qto properties linked to the element investigated.
    for name, properties in list(psets.items())[:3]: # names all yhr ptoperties, parameters and values
        print(f'\n{name}:', properties) 

Class: IfcWall
Name: Basic Wall:BA_Heavy Weight_Concrete_200mm:1794121
GlobalId: 2xflkD7IDEMgp51ZbDNMUT
Available property / quantity sets: ['Pset_WallCommon', 'Qto_WallBaseQuantities', 'Pset_EnvironmentalImpactIndicators', 'Pset_ReinforcementBarPitchOfWall', 'Qto_BodyGeometryValidation']

Pset_WallCommon: {'ThermalTransmittance': 5.23, 'id': 3453, 'Reference': 'BA_Heavy Weight_Concrete_200mm', 'IsExternal': True, 'LoadBearing': False, 'ExtendToStructure': False}

Qto_WallBaseQuantities: {'Height': 3509.999999999999, 'Length': 10229.999999999927, 'Width': 200.0, 'GrossFootprintArea': 2.046000000000229, 'GrossVolume': 7.181459999999948, 'GrossSideArea': 35.907299999999736, 'NetSideArea': 35.907299999999736, 'NetVolume': 7.181459999999557, 'id': 3441}

Pset_EnvironmentalImpactIndicators: {'Reference': 'BA_Heavy Weight_Concrete_200mm', 'id': 3447}


## 5. Find element locations

Another IfcOpenShell function is `get_container()`. 
This function returns an element’s assigned spatial container, when available, for a given element.

In [ ]:
rows = []
for element in elements: # Runs a command that expands on the previous properties code, but now additionally appends a collumn with the name of the spacial container the investigated elements are in.
    container = ifcopenshell.util.element.get_container(element)
    rows.append({
        "GlobalId": element.GlobalId,
        "Name": element.Name,
        "IFC class": element.is_a(),
        "Container": container.Name if container else None,
    })

element_df = pd.DataFrame(rows, columns=["GlobalId", "Name", "IFC class", "Container"])
display(element_df.head(10))


,GlobalId,Name,IFC class,Container
0,2xflkD7IDEMgp51ZbDNMUT,Basic Wall:BA_Heavy Weight_Concrete_200mm:1794121,IfcWall,ES_Stue
1,2xflkD7IDEMgp51ZbDNMRi,Basic Wall:BA_Heavy Weight_Concrete_350mm:1794360,IfcWall,ES_Stue
2,2xflkD7IDEMgp51ZbDNMMm,Basic Wall:BA_Heavy Weight_Concrete_350mm:1794660,IfcWall,ES_Stue
3,2xflkD7IDEMgp51ZbDNMFK,Basic Wall:IN_Light Weight_110mm:1795072,IfcWall,ES_Stue
4,2xflkD7IDEMgp51ZbDNM8x,Basic Wall:IN_Light Weight_110mm:1795567,IfcWall,ES_Stue
5,2xflkD7IDEMgp51ZbDNM7i,Basic Wall:IN_Light Weight_110mm:1795640,IfcWall,ES_Stue
6,2xflkD7IDEMgp51ZbDNM5V,Basic Wall:IN_Light Weight_110mm:1795723,IfcWall,ES_Stue
7,2xflkD7IDEMgp51ZbDNM0y,Basic Wall:IN_Light Weight_110mm:1796072,IfcWall,ES_Stue
8,2xflkD7IDEMgp51ZbDNMzb,Basic Wall:IN_Light Weight_110mm:1796273,IfcWall,ES_Stue
9,2xflkD7IDEMgp51ZbDNMxH,Basic Wall:IN_Light Weight_110mm:1796357,IfcWall,ES_Stue


## 6. Export a table

For summarisation of the computed data collected through the IfcOpenShell ecxtension for python, the function outputs can be saved as a tabular CSV file.

In [ ]:
output_dir = Path("outputs") # Defines the path
output_dir.mkdir(exist_ok=True)
element_df.to_csv(output_dir / "elements.csv", index=False)
print("Saved:", output_dir / "elements.csv") # Svaes the CSV file
